# Model Training

## Import necessary libraries

In [1]:
%pip install -qq -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Add current directory to Python path for imports
import os
import sys

# Add the parent directory (project root) to Python path so we can import from src
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
# Utility Functions
from src.utils import create_spark_session

# Create Spark session
spark, sedona = create_spark_session(app_name="ModelTrainingSpark")

## Loading Datasets

In [4]:
from src.utils import read_config_path

# Load data using configuration file
filepath = read_config_path(key="model_training_data_path", domain="processed")

df_raw = spark.read.csv(
    filepath,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"',
)

In [5]:
from src.utils import preprocessed_data_converter

df_prepared = preprocessed_data_converter(df_raw)

In [6]:
df_prepared.show(5, truncate=False)

+---------------+--------------+---------------+----------------------------+--------------------+-------------------------+--------------------+
|timestamp_month|timestamp_year|resolution_time|address_encoded             |latlong_encoded     |organization_encoded     |type_encoded        |
+---------------+--------------+---------------+----------------------------+--------------------+-------------------------+--------------------+
|9              |2021          |275            |(2048,[834,1804],[1.0,1.0]) |[13.67891,100.66709]|(1786,[10,53],[1.0,1.0]) |(25,[7,8],[1.0,1.0])|
|9              |2021          |253            |(2048,[348,426],[1.0,1.0])  |[13.7206,100.52649] |(1786,[49],[1.0])        |(25,[14],[1.0])     |
|12             |2021          |246            |(2048,[802,1656],[1.0,1.0]) |[13.8228,100.59165] |(1786,[31,108],[1.0,1.0])|(25,[0,8],[1.0,1.0])|
|12             |2021          |456            |(2048,[802,1656],[1.0,1.0]) |[13.8091,100.59131] |(1786,[31,172],[1.0,1.0])|

In [7]:
df_prepared.printSchema()

root
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- resolution_time: integer (nullable = true)
 |-- address_encoded: vector (nullable = true)
 |-- latlong_encoded: vector (nullable = true)
 |-- organization_encoded: vector (nullable = true)
 |-- type_encoded: vector (nullable = true)



---

## Model Training

### Sampling the preprocessed data

In [8]:
from src.utils import preprocessed_data_sampler

train_df, test_df = preprocessed_data_sampler(df_prepared, samplng_fraction=0.1)

---

### Gradient Boosted Tree Regressor Model

In [ ]:
from pyspark.ml.regression import GBTRegressor

from src.pipelines_spark import ModelDefinePipelineSpark


gbt = GBTRegressor(
    labelCol="resolution_time",
    featuresCol="features",
)

gbt_model_pipeline = ModelDefinePipelineSpark(
    name="GradientBoostedTree",
    model=gbt,
    input_columns=[
        "timestamp_month",
        "timestamp_year",
        "address_encoded",
        "latlong_encoded",
        "organization_encoded",
        "type_encoded",
    ],
    label_column="resolution_time",
    evaluators=["rmse", "mae", "r2"],
    param_dict={
        "maxDepth": [3, 5],
        "maxIter": [50, 100],
        "stepSize": [0.05],
    },
)

In [10]:
gbt_model_pipeline.fit(train_df, save_name="gbt")

✔ Model saved successfully to: C:\Users\pun\Desktop\CU\CEDT-Y2-S1\2110403 - Introduction to Data Science and Data Engineering\CEDT-2110403-DSDE-Project\data\model\gbt_cv_model_spark


CrossValidatorModel_3234d2dd45db

In [11]:
gbt_model_pipeline.evaluate(test_df)


===== CrossValidatorModel Results =====
RMSE: 89.73851485008632
MAE: 52.70711387403336
R2: 0.544364251274722


---

### Random Forest Regressor Model

In [12]:
from pyspark.ml.regression import RandomForestRegressor

from src.pipelines_spark import ModelDefinePipelineSpark


rf = RandomForestRegressor(
    labelCol="resolution_time",
    featuresCol="features",
)

rf_model_pipeline = ModelDefinePipelineSpark(
    name="RandomForestRegressor",
    model=rf,
    input_columns=[
        "timestamp_month",
        "timestamp_year",
        "address_encoded",
        "latlong_encoded",
        "organization_encoded",
        "type_encoded",
    ],
    label_column="resolution_time",
    evaluators=["rmse", "mae", "r2"],
    param_dict={
        "numTrees": [100],
        "maxDepth": [8, 12],
    },
)

In [13]:
rf_model_pipeline.fit(train_df, save_name="rf")

✔ Model saved successfully to: C:\Users\pun\Desktop\CU\CEDT-Y2-S1\2110403 - Introduction to Data Science and Data Engineering\CEDT-2110403-DSDE-Project\data\model\rf_cv_model_spark


CrossValidatorModel_b56b207e45dd

In [14]:
rf_model_pipeline.evaluate(test_df)


===== CrossValidatorModel Results =====
RMSE: 91.89305938421174
MAE: 54.333780898344145
R2: 0.5222227605244749


---

## Stop Spark

In [15]:
spark.stop()

---